# The Arithmetic Kakeya Conjecture

In [ ]:
#@title Verification code

import numpy as np
from scipy import optimize
import collections

minimize = optimize.minimize

def calculate_entropy(probabilities: np.ndarray) -> float:
  """Calculates Shannon entropy of a discrete distribution."""
  entropy = 0.0
  for p in probabilities:
    if p > 0:
      entropy -= p * np.log2(p)
  return entropy


def evaluate_sum_diff_conjecture(
    joint_prob_xy: np.ndarray, slope_tuple: Tuple[int, int]
) -> float:
  """Evaluates Sum-Difference Conjecture ratio for given joint distribution."""
  slopes_to_avoid = [(0, 1), (1, 0), slope_tuple]
  if joint_prob_xy.ndim != 2:  # Check if it's a 2D matrix
    return 0.0

  if joint_prob_xy.shape[0] != joint_prob_xy.shape[1]:  # Check if square
    return 0.0
  n_values = joint_prob_xy.shape[0]  # Infer n_values from matrix size
  if n_values < 2:
    return 0.0

  values = np.arange(1, n_values + 1)
  x_values = values
  y_values = values

  # --- Robustness Checks ---
  # 1. Clip negative values to zero and ensure non-negativity
  joint_prob_xy = np.clip(joint_prob_xy, 0, None)  # Clip to be >= 0

  # 2. Renormalize joint_prob_xy to ensure sum is 1
  if np.sum(joint_prob_xy) > 0:  # Avoid division by zero if all become zero
    joint_prob_xy /= np.sum(joint_prob_xy)
  else:
    return 0.0  # If all probs are zero, return score 0

  # 3. Calculate marginal distributions (after clipping and renormalizing)
  prob_x = np.sum(
      joint_prob_xy, axis=1
  )  # Marginal distribution for X (row sums)
  prob_y = np.sum(
      joint_prob_xy, axis=0
  )  # Marginal distribution for Y (column sums)

  distinct_diffs, fixed_joint_prob_xy = ensure_distinct_differences(
      joint_prob_xy, x_values, y_values
  )
  if not distinct_diffs:
    joint_prob_xy = fixed_joint_prob_xy  # Update with fixed version

    # Re-calculate marginals AGAIN after distinct difference fix and
    # renormalization
    prob_x = np.sum(fixed_joint_prob_xy, axis=1)
    prob_y = np.sum(fixed_joint_prob_xy, axis=0)

    # Renormalize AGAIN after ensure_distinct_differences
    if np.sum(joint_prob_xy) > 0:  # avoid divide by zero
      joint_prob_xy /= np.sum(joint_prob_xy)
    else:
      return 0.0

  prob_x_nonzero = prob_x[prob_x > 0]
  prob_y_nonzero = prob_y[prob_y > 0]

  if not np.any(prob_x_nonzero) or not np.any(prob_y_nonzero):
    return 0.0  # Avoid errors if all probabilities become zero

  entropy_xy = calculate_joint_entropy(joint_prob_xy)

  max_entropy_value = 0.0
  for slope in slopes_to_avoid:
    entropy_slope = calculate_slope_entropy(
        slope, joint_prob_xy, x_values, y_values
    )
    max_entropy_value = max(max_entropy_value, entropy_slope)

  denominator = max_entropy_value
  if denominator == 0:
    return 0.0  # Avoid division by zero

  ratio = entropy_xy / denominator
  return ratio


def evaluate_sum_diff_conjecture_hidden(
    joint_prob_xy: np.ndarray, slope_tuple: Tuple[int, int]
) -> float:
  """Evaluates Sum-Difference Conjecture ratio for given joint distribution."""
  slopes_to_avoid = [(0, 1), (1, 0), slope_tuple]
  if joint_prob_xy.ndim != 2:  # Check if it's a 2D matrix
    return 0.0

  if joint_prob_xy.shape[0] != joint_prob_xy.shape[1]:  # Check if square
    return 0.0
  n_values = joint_prob_xy.shape[0]  # Infer n_values from matrix size
  if n_values < 2:
    return 0.0

  values = np.arange(1, n_values + 1)
  x_values = values
  y_values = values

  # --- Robustness Checks ---
  # 1. Clip negative values to zero and ensure non-negativity
  joint_prob_xy = np.clip(joint_prob_xy, 0, None)  # Clip to be >= 0

  # 2. Renormalize joint_prob_xy to ensure sum is 1
  if np.sum(joint_prob_xy) > 0:  # Avoid division by zero if all become zero
    joint_prob_xy /= np.sum(joint_prob_xy)
  else:
    return 0.0  # If all probs are zero, return score 0

  # 3. Calculate marginal distributions (after clipping and renormalizing)
  prob_x = np.sum(
      joint_prob_xy, axis=1
  )  # Marginal distribution for X (row sums)
  prob_y = np.sum(
      joint_prob_xy, axis=0
  )  # Marginal distribution for Y (column sums)

  distinct_diffs, fixed_joint_prob_xy = ensure_distinct_differences(
      joint_prob_xy, x_values, y_values
  )
  if not distinct_diffs:
    joint_prob_xy = fixed_joint_prob_xy  # Update with fixed version

    # Re-calculate marginals AGAIN after distinct difference fix and
    # renormalization
    prob_x = np.sum(fixed_joint_prob_xy, axis=1)
    prob_y = np.sum(fixed_joint_prob_xy, axis=0)

    # Renormalize AGAIN after ensure_distinct_differences
    if np.sum(joint_prob_xy) > 0:  # avoid divide by zero
      joint_prob_xy /= np.sum(joint_prob_xy)
    else:
      return 0.0

  prob_x_nonzero = prob_x[prob_x > 0]
  prob_y_nonzero = prob_y[prob_y > 0]

  if not np.any(prob_x_nonzero) or not np.any(prob_y_nonzero):
    return 0.0  # Avoid errors if all probabilities become zero

  entropy_xy = calculate_joint_entropy(joint_prob_xy)

  max_entropy_value = 0.0
  for slope in slopes_to_avoid:
    entropy_slope = calculate_slope_entropy(
        slope, joint_prob_xy, x_values, y_values
    )
    max_entropy_value = max(max_entropy_value, entropy_slope)

  denominator = max_entropy_value
  if denominator == 0:
    return 0.0  # Avoid division by zero

  ratio = entropy_xy / denominator
  return ratio


def ensure_distinct_differences(joint_prob_xy, x_values, y_values):
  """Ensures X-Y differences are distinct by modifying joint_prob_XY."""
  diff_counts = collections.defaultdict(list)

  # 1. Collect differences and joint probabilities (from joint_prob_xy)
  for i, x in enumerate(x_values):
    for j, y in enumerate(y_values):
      if joint_prob_xy[i, j] > 0:
        diff = x - y
        diff_counts[diff].append(
            ((i, j), joint_prob_xy[i, j])
        )  # Store indices (i,j) and joint prob

  duplicates_found = False
  fixed_joint_prob_xy = joint_prob_xy.copy()

  for _, instances in diff_counts.items():
    if len(instances) > 1:  # Duplicates found
      duplicates_found = True
      probs_to_fix = sorted(instances, key=lambda item: item[1], reverse=True)
      highest_prob_indices = probs_to_fix[0][0]  # (i,j) of highest joint prob

      for k in range(1, len(probs_to_fix)):
        (x_index_to_zero, y_index_to_zero), _ = probs_to_fix[k]

        # 2. Heuristic fix (modify joint_prob_XY directly)
        # Need to decide how to redistribute probability within joint_prob_XY
        # Option 1: Zero out joint_prob_XY[x_index_to_zero, y_index_to_zero]
        #          and add to joint_prob_XY[highest_prob_indices]

        removed_prob = fixed_joint_prob_xy[x_index_to_zero, y_index_to_zero]
        fixed_joint_prob_xy[x_index_to_zero, y_index_to_zero] = 0.0
        fixed_joint_prob_xy[
            highest_prob_indices
        ] += removed_prob  # Add to highest prob instance

      fixed_joint_prob_xy /= np.sum(
          fixed_joint_prob_xy
      )  # Renormalize joint_prob_xy

  return (
      not duplicates_found,
      fixed_joint_prob_xy,
  )  # Return modified joint_prob_xy


def calculate_joint_entropy(joint_prob_xy):  # Takes joint_prob_xy
  """Calculates joint entropy H(X,Y) from joint probability matrix."""
  joint_entropy = 0.0
  for i in range(joint_prob_xy.shape[0]):
    for j in range(joint_prob_xy.shape[1]):
      p_joint = joint_prob_xy[i, j]  # Use joint probability directly
      if p_joint > 0:
        joint_entropy -= p_joint * np.log2(p_joint)
  return joint_entropy


def calculate_slope_entropy(slope, joint_prob_xy, x_values, y_values):
  """Calculates entropy H(X+Y) from joint probability matrix."""
  sum_counts = collections.defaultdict(float)
  for i, x in enumerate(x_values):
    for j, y in enumerate(y_values):
      p_joint = joint_prob_xy[i, j]
      if p_joint > 0:
        sum_val = slope[0] * x + slope[1] * y
        sum_counts[sum_val] += p_joint

  prob_sum = np.array(list(sum_counts.values()))
  entropy_sum = calculate_entropy(prob_sum)
  return entropy_sum


) -> tuple[dict[str, float], dict[str, str]]:
  """Returns the ratio for the Sum-Difference Conjecture using joint distribution."""
  result = {}
  feedback = {}

  best_joint_prob_xy_result = search_for_best_distributions(slope_tuple)
  score = evaluate_sum_diff_conjecture_hidden(
      best_joint_prob_xy_result, slope_tuple
  )

  n_values = best_joint_prob_xy_result.shape[0]
  values = np.arange(1, n_values + 1)
  x_values = values
  y_values = values
  joint_prob_xy = np.clip(best_joint_prob_xy_result, 0, None)  # Clip to be >= 0
  joint_prob_xy /= np.sum(joint_prob_xy)
  _, fixed_joint_prob_xy = ensure_distinct_differences(
      joint_prob_xy, x_values, y_values
  )
  result['score'] = score
  feedback['best_joint_prob_xy'] = fixed_joint_prob_xy
  feedback['best_ratio_found'] = score
  feedback['slope'] = str(slope_tuple)
  feedback = format_feedback_repr(feedback)
  return result, feedback

In [ ]:
#@title Initial program

"""Finds joint probability distributions to test Sum-Difference Conjecture."""
import itertools
import logging
import time
from scipy import integrate
import numpy as np
from scipy import optimize
import warnings
import re
from typing import Any, Callable, Mapping, List, Tuple
import scipy.linalg as la
import numpy.polynomial.polynomial as poly
import collections

minimize = optimize.minimize

def search_for_best_distributions(
    slope_to_avoid: Tuple[int, int],
) -> np.ndarray:
  """Searches for joint probability distributions maximizing the ratio."""
  a, b = slope_to_avoid
  best_joint_prob_xy = np.random.uniform(0, 1, (83, 83))
  # zero out all (x, y) entries where x is not divisible by a or y is not
  # divisible by a
  y_not_divisible_by_a = np.arange(83) % a != 0
  x_not_divisible_by_b = np.arange(83) % b != 0

  mask_83 = (
      x_not_divisible_by_b[:, np.newaxis] | y_not_divisible_by_a[np.newaxis, :]
  )
  best_joint_prob_xy[mask_83] = 0.0

  # Initialize joint_prob_xy with best joint_prob_xy found so far, from above
  # This is to ensure that the search starts from a good starting point
  # But we don't want to always do this, to avoid being stuck in local optima
  variable_name = f'best_joint_prob_xy_{slope_to_avoid[0]}_{slope_to_avoid[1]}'
  if np.random.rand() < 0.5 and variable_name in globals():
    best_joint_prob_xy = globals()[variable_name]

  joint_prob_xy = best_joint_prob_xy.copy()
  joint_prob_xy /= np.sum(joint_prob_xy)

  best_score = evaluate_sum_diff_conjecture(joint_prob_xy, slope_to_avoid)
  start_time = time.time()
  eval_count = 0

  while time.time() - start_time < 100:  # Search for 100 seconds
    # Greedy search: adjust a random cell and keep the change if it
    # improves the score

    # We only consider cells (x, y) where x is divisible by b and y is divisible
    # by a

    row_index = np.random.randint((joint_prob_xy.shape[0] // b) * b)
    col_index = np.random.randint((joint_prob_xy.shape[1] // a) * a)
    joint_prob_xy[row_index, col_index] += np.random.normal(0, 0.05)
    joint_prob_xy = np.maximum(joint_prob_xy, 0)
    joint_prob_xy /= np.sum(joint_prob_xy)

    score = evaluate_sum_diff_conjecture(joint_prob_xy, slope_to_avoid)
    eval_count += 1
    if score > best_score:
      best_score = score
      best_joint_prob_xy = joint_prob_xy.copy()
      print(f'Improved SD Ratio (Joint): {score:.4f}')

    if np.random.rand() < 0.5:
      joint_prob_xy = best_joint_prob_xy.copy()

    if np.random.rand() < 0.2:
      joint_prob_xy = np.random.uniform(0, 1, (83, 83))
      joint_prob_xy[mask_83] = 0.0

  return best_joint_prob_xy




**Prompt used**

Act as an expert software developer and optimization specialist specializing in creating probability distributions with certain properties.
Your task is to generate a matrix, which maximizes the following evaluation function:

@njit
def evaluate_sum_diff_conjecture_numba_part(
    joint_prob_xy, slopes_to_avoid, slope_tuple
):
  n_values = joint_prob_xy.shape[0]
  values = np.arange(1, n_values + 1)
  x_values = values
  y_values = values
  joint_prob_xy = np.clip(joint_prob_xy, 0, None)
  if np.sum(joint_prob_xy) > 0:
    joint_prob_xy /= np.sum(joint_prob_xy)
  else:
    return 0.0

lhs = calculate_slope_entropy(slope_tuple, joint_prob_xy, x_values, y_values)

max_entropy_value = 0.0
  for slope in slopes_to_avoid:
    entropy_slope = calculate_slope_entropy(
        slope, joint_prob_xy, x_values, y_values
    )
    max_entropy_value = max(max_entropy_value, entropy_slope)

denominator = max_entropy_value
  if denominator == 0:
    return 0.0  # Avoid division by zero

ratio = lhs / denominator
  return ratio

def evaluate_sum_diff_conjecture(
    joint_prob_xy, slope_tuple: Tuple[int, int]
) -> float:
  """Evaluates Sum-Difference Conjecture ratio for given joint distribution."""
  slopes_to_avoid = [(0, 1), (1, 0), (1, 2)]

return evaluate_sum_diff_conjecture_numba_part(
      joint_prob_xy, slopes_to_avoid, slope_tuple
  )

Your task is to write a search function that searches for the best matrix. Gaussians are a good baseline, but you probably should experiment with other, more general distributions (and then find the best parameters to make them work). Your function will have 1000 seconds to run, and after that it has to have returned the best lists it found. If after 1000 seconds it has not returned anything, it will be terminated with negative infinity points. All numbers in your matrix have to be positive or zero.

You may code up any search method you want, and you are allowed to call the evaluate_sum_diff_conjecture() function as many times as you want. You have access to it, you don't need to code up the evaluate_sum_diff_conjecture() function.

In [ ]:
#@title Example code evolved by AlphaEvolve


def search_for_best_distributions(
    slope_to_avoid: Tuple[int, int],
) -> np.ndarray:
  """Searches for joint probability distributions maximizing the ratio."""
  a, b = slope_to_avoid
  """Searches for joint probability distributions maximizing the ratio."""
  n_values = 70
  best_joint_prob_xy = np.random.uniform(0, 1, (n_values, n_values))
  variable_name = f'best_joint_prob_xy_{slope_to_avoid[0]}_{slope_to_avoid[1]}'
  if np.random.rand() < 0.5 and variable_name in globals():
    best_joint_prob_xy = globals()[variable_name]

  joint_prob_xy = best_joint_prob_xy.copy()
  joint_prob_xy /= np.sum(joint_prob_xy)

  best_score = evaluate_sum_diff_conjecture(joint_prob_xy, slope_to_avoid)
  start_time = time.time()
  x = np.linspace(0, n_values, n_values)
  y = np.linspace(0, n_values, n_values)
  xv, yv = np.meshgrid(x, y)
  positions = np.stack((xv, yv), axis=-1)

  def objective_function_single(params):
    mean1 = params[:2]
    cov1 = np.array([[params[2], params[3]], [params[3], params[4]]])

    if not is_positive_definite_2x2(cov1):
      return float('inf')

    rv1 = multivariate_normal(mean1, cov1)
    joint_prob_xy = rv1.pdf(positions)
    joint_prob_xy = np.maximum(joint_prob_xy, 0)
    joint_prob_xy /= np.sum(joint_prob_xy)

    score = evaluate_sum_diff_conjecture(joint_prob_xy, slope_to_avoid)
    return -score
  bounds_single = [
      (0, n_values),  # mean1_x
      (0, n_values),  # mean1_y
      (0.1, 10),    # cov1_xx
      (-5, 5),      # cov1_xy
      (0.1, 10),    # cov1_yy
  ]

  result = optimize.differential_evolution(objective_function_single, bounds_single, maxiter=5, popsize=5, tol=0.01, mutation=(0.5, 1), recombination=0.7, seed=42)
  x = np.linspace(0, n_values, n_values)
  y = np.linspace(0, n_values, n_values)
  xv, yv = np.meshgrid(x, y)
  positions = np.stack((xv, yv), axis=-1)

  def gaussian_objective(params, num_gaussians, current_joint_prob_xy=None):
    joint_prob_xy = np.zeros((n_values, n_values)) if current_joint_prob_xy is None else current_joint_prob_xy
    weight_sum = 0.0

    for i in range(num_gaussians):
      mean = params[i*5 : i*5 + 2]
      cov = np.array([[params[i*5 + 2], params[i*5 + 3]], [params[i*5 + 3], params[i*5 + 4]]])

      if not is_positive_definite_2x2(cov):
         return float('inf')

      rv = multivariate_normal(mean, cov)
      if num_gaussians == 1:
         weight = 1.0
      elif i==0:
        weight = params[5*num_gaussians]
      else:
        weight = 1.0 - params[5*num_gaussians]

      joint_prob_xy += weight * rv.pdf(positions)
      if num_gaussians > 1:
        weight_sum += weight

    # perturbation fallback - add small gaussian noise when we are doing a full search
    if current_joint_prob_xy is None:
      row = np.random.randint(0, n_values)
      col = np.random.randint(0, n_values)
      joint_prob_xy[row, col] += np.random.normal(0, 0.02)


    joint_prob_xy = np.maximum(joint_prob_xy, 0)
    joint_prob_xy /= np.sum(joint_prob_xy)

    score = evaluate_sum_diff_conjecture(joint_prob_xy, slope_to_avoid)
    return -score


  bounds_single = [
      (0, n_values),  # mean1_x
      (0, n_values),  # mean1_y
      (0.1, 10),    # cov1_xx
      (-5, 5),      # cov1_xy
      (0.1, 10),    # cov1_yy
  ]

  bounds_double = [
      (0, n_values),  # mean1_x
      (0, n_values),  # mean1_y
      (0.1, 10),    # cov1_xx
      (-5, 5),      # cov1_xy
      (0.1, 10),    # cov1_yy
      (0.001, 0.999), # weight1
      (0, n_values),  # mean2_x
      (0, n_values),  # mean2_y
      (0.1, 10),    # cov2_xx
      (-5, 5),      # cov2_xy
      (0.1, 10),    # cov2_yy
  ]

  bounds = [bounds_single, bounds_double]

  for num_gaussians in range(1, 3):

    if num_gaussians == 1:
        nparams = 5
    elif num_gaussians == 2 :
        nparams = 11

    current_bounds = bounds_single if num_gaussians == 1 else bounds_double

    result = optimize.differential_evolution(
      lambda params: gaussian_objective(params, num_gaussians),
        current_bounds,
        maxiter=max(5, int(30 * (990 - (time.time() - start_time)) / 990)),
        popsize=max(5, int(20 * (990 - (time.time() - start_time)) / 990)),
      tol=0.01,
      mutation=(0.5, 1),
      recombination=0.7,
      seed=42
    )

    if result.success:
        params = result.x
        score  = -gaussian_objective(params, num_gaussians)
        if score > best_score:
          best_score = score
          best_joint_prob_xy = ((-1)*gaussian_objective(params, num_gaussians, np.zeros((n_values, n_values)),  ) )
          print(f'Improved SD Ratio (Joint, {num_gaussians} Gaussian): {score:.4f}')

    if time.time() - start_time > 990:
        break

  return best_joint_prob_xy


## What AlphaEvolve found

For the 3-slope case $C(\{0,1,\infty\}; -1)$, AlphaEvolve improved the lower bound only in the eighth decimal. For the 4-slope case, AlphaEvolve obtained the more interesting improvement $C(\{0,1,2,\infty\};-1) \geq 1.668$, up from the previously known lower bound of $1.61226$. The joint distributions of the random variables found by AlphaEvolve resembled discrete Gaussians. Inspired by the form of these constructions, the third author was able to establish rigorous asymptotics for $C(\{0,1,\infty\}; a/b)$, proving that $C \approx 2 - \Theta(1/\log(|a|+|b|))$.